### Preprocessing Audio Cycles
This script groups rows by filename to load each `.wav` file only once. This vastly speeds up processing compared to loading the file for every individual cycle.

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfilt

# Configuration
data_dir = r'C:\Users\Kruthi K Shetty\data-science\data'
audio_dir = os.path.join(data_dir, 'audio_and_txt_files')
df_path = os.path.join(data_dir, 'cycles_df.csv')
output_npy = os.path.join(data_dir, 'processed_cycles.npy')
output_df = os.path.join(data_dir, 'cycles_df_processed.csv')

SR = 22050
TARGET_DUR = 3.0
TARGET_SAMPLES = int(SR * TARGET_DUR)

# Bandpass filter design
def butter_bandpass(lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    sos = butter(order, [low, high], btype='band', output='sos')
    return sos

def bandpass_filter(data, lowcut, highcut, fs, order=5):
    sos = butter_bandpass(lowcut, highcut, fs, order=order)
    y = sosfilt(sos, data)
    return y

# 1. Load DataFrame
print(f"Loading {df_path}...")
try:
    df = pd.read_csv(df_path)
except Exception as e:
    print(f"Could not load dataframe: {e}")
    df = pd.DataFrame() # Empty to prevent crash

processed_clips = []
valid_indices = []

skipped = 0
processed = 0

sample_plots = {
    'Normal': None,
    'Wheeze': None,
    'Crackle': None
}

# Group by filename to avoid loading the same audio file multiple times
# This is MUCH faster!
grouped = df.groupby('filename')
print(f"Total unique audio files to process: {len(grouped)}")

for filename_txt, group in grouped:
    filename_wav = str(filename_txt).replace('.txt', '.wav')
    wav_path = os.path.join(audio_dir, filename_wav)
    
    if not os.path.exists(wav_path):
        skipped += len(group)
        print(f"Skipped {filename_wav}: File not found")
        continue
        
    try:
        # Load the ENTIRE audio file once and resample to 22050 Hz
        y_full, sr = librosa.load(wav_path, sr=SR)
        
        for idx, row in group.iterrows():
            start_time = float(row['cycle_start'])
            end_time = float(row['cycle_end'])
            label = row['label']
            
            start_sample = int(start_time * SR)
            end_sample = int(end_time * SR)
            
            # Slice from the pre-loaded audio
            y = y_full[start_sample:end_sample]
            
            # Keep original for plot if needed
            y_original = y.copy()
            
            # Trim or zero-pad
            if len(y) > TARGET_SAMPLES:
                y = y[:TARGET_SAMPLES]
            elif len(y) < TARGET_SAMPLES:
                y = np.pad(y, (0, TARGET_SAMPLES - len(y)), mode='constant')
                
            # Bandpass filter
            y_filtered = bandpass_filter(y, 100, 2000, SR)
            
            # Save sample for plotting
            if label in sample_plots and sample_plots[label] is None:
                # Pad original too for parallel plotting
                if len(y_original) > TARGET_SAMPLES:
                    y_original = y_original[:TARGET_SAMPLES]
                else:
                    y_original = np.pad(y_original, (0, TARGET_SAMPLES - len(y_original)), mode='constant')
                sample_plots[label] = (y_original, y_filtered)
                
            processed_clips.append(y_filtered)
            valid_indices.append(idx)
            processed += 1
            
    except Exception as e:
        skipped += len(group)
        print(f"Skipped {filename_wav}: Error processing - {e}")

print(f"\nTotal clips processed: {processed}")
print(f"Total clips skipped: {skipped}")

if processed > 0:
    # 3. Store processed clips as numpy array
    npy_array = np.array(processed_clips)
    print(f"Shape of the saved numpy array: {npy_array.shape}")
    np.save(output_npy, npy_array)

    # 4. Save updated DataFrame
    df_processed = df.loc[valid_indices].copy()
    df_processed['npy_index'] = np.arange(len(df_processed))
    df_processed.to_csv(output_df, index=False)
    print(f"Saved processed DataFrame to {output_df}")

    # 5. Plotting
    fig, axes = plt.subplots(3, 2, figsize=(15, 10), sharex=True, sharey=True)
    fig.suptitle('Waveforms Before and After Bandpass Filter (100-2000Hz)', fontsize=16)

    time_axis = np.linspace(0, TARGET_DUR, TARGET_SAMPLES)

    plot_labels = ['Normal', 'Wheeze', 'Crackle']

    for i, lbl in enumerate(plot_labels):
        if lbl in sample_plots and sample_plots[lbl] is not None:
            y_orig, y_filt = sample_plots[lbl]
            
            # Before
            axes[i, 0].plot(time_axis, y_orig, color='gray')
            axes[i, 0].set_title(f'{lbl} - Before Filter')
            axes[i, 0].set_ylabel('Amplitude')
            
            # After
            axes[i, 1].plot(time_axis, y_filt, color='blue')
            axes[i, 1].set_title(f'{lbl} - After Filter')

    axes[2, 0].set_xlabel('Time (s)')
    axes[2, 1].set_xlabel('Time (s)')
    plt.tight_layout()
    plt.show()
else:
    print("No clips were successfully processed.")
